In [1]:
import os
from pyspark.sql import functions as F

def write_to_gold(df, output_path: str, merge_keys: list):
    """
    Local merge logic for Gold layer.
    """

    df = (
        df.withColumn("created_timestamp", F.current_timestamp())
          .withColumn("updated_timestamp", F.current_timestamp())
    )

    if os.path.exists(output_path):
        existing_df = spark.read.parquet(output_path)
        merged_df = (
            existing_df.unionByName(df)
                       .dropDuplicates(merge_keys)
        )
    else:
        merged_df = df

    merged_df.write.mode("overwrite").parquet(output_path)
